In [1]:
'''
Note: built this incrementally and only realized midway through that a dataclass
structure would've made it way more readable. didn't have time to refactor,
so fair warning. population size capped at 500 across all runs — see README for results.
'''

"\nNote: built this incrementally and only realized midway through that a dataclass\nstructure would've made it way more readable. didn't have time to refactor,\nso fair warning. population size capped at 500 across all runs — see README for results.\n"

In [ ]:
!pip install cma

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import cma

In [ ]:
num_dim = 2
num_candidates = 500
tau = 1/math.sqrt(num_dim)

candidates = [[random.uniform(-500, 500) for x in range(num_dim)] for i in range(num_candidates)]

max_ind = max(max(individual) for individual in candidates)
min_ind = min(min(individual) for individual in candidates)

single_sigma = 0.1 * (max_ind - min_ind)

sigma_per_dim = [0.1 * (max_ind - min_ind)]*num_dim

sigma_per_dim_per_ind = []
for i in candidates:
    s = [0.1 * (max(i) - min(i))]*num_dim
    sigma_per_dim_per_ind.append(s)

candidate_sigma_list = []
for i in range(num_candidates):
    x = [candidates[i], sigma_per_dim_per_ind[i]]
    candidate_sigma_list.append(x)

In [ ]:
def schwefel(num_dim, candidate):
    sum_ =  0
    for x in candidate:
        sum_ += x * math.sin(math.sqrt(abs(x)))
    value = 418.9829 * num_dim - sum_
    return value

def fitness(num_dim, candidate):
    value = schwefel(num_dim, candidate)
    return -value

def fitness_candidate_sigma(num_dim, candidate):
    value = schwefel(num_dim, candidate[0])
    return -value

In [ ]:
# Mutation part for global single sigma system

def single_sigma_update(sigma, tau, success=True):
    z = random.gauss(0.0, 1.0)
    if success:
        return sigma * math.exp(tau*z)
    else:
        return sigma * math.exp(-tau*z)

def x_update(candidate, sigma):
    updated_x = [x + random.gauss(0, sigma) for x in candidate]
    for i, v in enumerate(updated_x):
        if v>500:
            updated_x[i] = 500
        elif v< -500:
            updated_x[i] = -500
    return updated_x

In [ ]:
# Mutation part for global per dimensional sigma system

def sigma_per_dim_update(sigmas, tau, success=True):
    if success:
        return [sigma * math.exp(tau * random.gauss(0.0,1.0)) for sigma in sigmas]
    else:
        return [sigma * math.exp(-tau * random.gauss(0.0,1.0)) for sigma in sigmas]

def x_update_sigma_per_dim(candidate, sigmas):
    updated_x = []
    for i in range(num_dim):
        x = candidate[i] + random.gauss(0.0, sigmas[i])
        updated_x.append(x)
    for i, v in enumerate(updated_x):
        if v>500: updated_x[i] = 500
        elif v< -500: updated_x[i] = -500
    return updated_x

In [ ]:
# Mutation part for per dimensional sigma for each candidate system

def candidate_sigma_update(candidate, tau):
    coordinates, sigmas = candidate[0], candidate[1]
    updated_x = []
    for i in range(num_dim):
        x = coordinates[i] + random.gauss(0.0, sigmas[i])
        updated_x.append(x)
    for i, v in enumerate(updated_x):
        if v>500: updated_x[i] = 500
        elif v< -500: updated_x[i] = -500
    updated_sigma = [sigma * math.exp(tau * random.gauss(0.0,1.0)) for sigma in sigmas]
    return [updated_x, updated_sigma]

In [ ]:
def ES_comma_global(num_dim, candidates, sigma, sigma_type="single_sigma"):
    success_count = 0
    offsprings = []
    fitness_list = []
    total_offsprings = num_candidates * 7
    for x in range(total_offsprings):
        parent = random.choice(candidates)
        if sigma_type == "single_sigma":
            child = x_update(parent, sigma)
        elif sigma_type == "per_dim":
            child = x_update_sigma_per_dim(parent, sigma)
        child_fitness = fitness(num_dim, child)
        parent_fitness = fitness(num_dim, parent)
        if child_fitness > parent_fitness:
            success_count += 1
        offsprings.append(child)
        fitness_list.append(child_fitness)
    combined_list = sorted(zip(fitness_list, offsprings), reverse=True)
    fit, sorted_offsprings = zip(*combined_list)
    next_gen = sorted_offsprings[0:num_candidates]
    success_rate = success_count / total_offsprings
    success = True if success_rate>0.2 else False
    # success = True ###this is to check with and without 1/5th rule
    
    return next_gen, success

def ES_plus_global(num_dim, candidates, sigma, sigma_type="single_sigma"):
    success_count = 0
    offsprings = []
    fitness_list = []
    total_offsprings = num_candidates * 4
    for x in range(total_offsprings):
        parent = random.choice(candidates)
        if sigma_type == "single_sigma":
            child = x_update(parent, sigma)
        elif sigma_type == "per_dim":
            child = x_update_sigma_per_dim(parent, sigma)
        child_fitness = fitness(num_dim, child)
        parent_fitness = fitness(num_dim, parent)
        if child_fitness > parent_fitness:
            success_count += 1
        offsprings.append(child)
        fitness_list.append(child_fitness)
    next_gen = []
    for i in candidates:
        offsprings.append(i)
    for i in candidates:
        fit = fitness(num_dim, i)
        fitness_list.append(fit)
    combined_list = sorted(zip(fitness_list, offsprings), reverse=True)
    fit, sorted_offsprings = zip(*combined_list)
    next_gen = sorted_offsprings[0:num_candidates]
    success_rate = success_count / total_offsprings
    success = True if success_rate>0.2 else False
    # success = True ###this is to check with and without 1/5th rule
    
    return next_gen, success

In [ ]:
def ES_comma_candidate_sigma(num_dim, candidate_sigma_list):
    offsprings = []
    fitness_list = []
    total_offsprings = num_candidates * 7
    for x in range(total_offsprings):
        parent = random.choice(candidate_sigma_list)
        child = candidate_sigma_update(parent, tau)
        offsprings.append(child)
        child_fitness = fitness_candidate_sigma(num_dim, child)
        fitness_list.append(child_fitness)
    combined_list = sorted(zip(fitness_list, offsprings), reverse=True)
    fit, sorted_offsprings = zip(*combined_list)
    next_gen = sorted_offsprings[0:num_candidates]
    return next_gen

def ES_plus_candidate_sigma(num_dim, candidate_sigma_list):
    offsprings = []
    fitness_list = []
    total_offsprings = num_candidates * 4
    for x in range(total_offsprings):
        parent = random.choice(candidate_sigma_list)
        child = candidate_sigma_update(parent, tau)
        offsprings.append(child)
        child_fitness = fitness_candidate_sigma(num_dim, child)
        fitness_list.append(child_fitness)
    offsprings.extend(candidate_sigma_list)
    for i in candidate_sigma_list:
        fit = fitness_candidate_sigma(num_dim, i)
        fitness_list.append(fit)
    combined_list = sorted(zip(fitness_list, offsprings), reverse=True)
    fit, sorted_offsprings = zip(*combined_list)
    next_gen = sorted_offsprings[0:num_candidates]
    return next_gen

In [ ]:
def mean_sigma_per_dim(candidate_sigma_list):
    num_candidates = len(candidate_sigma_list)

    mean_sigmas = [0.0] * num_dim
    for candidate in candidate_sigma_list:
        sigmas = candidate[1]
        for d in range(num_dim):
            mean_sigmas[d] += sigmas[d]

    return [s / num_candidates for s in mean_sigmas]

In [ ]:
# uncomment one block at a time to run each variant

num_gen = 100
sigma_history = []
sigma_type = "per_dim"
for i in range(num_gen):

#     ##for single global sigma
#     # sigma_history.append(single_sigma)
#     # candidates, success = ES_plus_global(num_dim, candidates, single_sigma, sigma_type)
#     # single_sigma = single_sigma_update(single_sigma, tau, success)

#     # #for global sigma per dim
#     # sigma_history.append(sigma_per_dim)
#     # candidates, success = ES_plus_global(num_dim, candidates, sigma_per_dim, sigma_type)
#     # sigma_per_dim = sigma_per_dim_update(sigma_per_dim, tau, success)

#     #for per dim per ind
#     candidate_sigma_list = ES_plus_candidate_sigma(num_dim, candidate_sigma_list)
#     sigma_history.append(mean_sigma_per_dim(candidate_sigma_list))

# sigma_history_per_dim = list(map(list, zip(*sigma_history))) #only for per dim per ind

In [13]:
# #global sigma and sigma per dim output

# print(candidates[0])
# a = fitness(num_dim, candidates[0])
# print(a * -1)

# plt.plot(sigma_history)
# plt.xlabel("generation")
# plt.ylabel("step_sizes")
# plt.grid(True)
# plt.show()

In [14]:
# #sigma per dim per indi output

# print(candidate_sigma_list[0][0])
# a = fitness_candidate_sigma(num_dim, candidate_sigma_list[0])
# print(a*-1)

# for d, sigma_series in enumerate(sigma_history_per_dim):
#     plt.plot(sigma_series, label=f"dim {d}")

# plt.xlabel("Generation")
# plt.ylabel("Average sigma")
# plt.legend()
# plt.grid(True)
# plt.show()

In [ ]:
##because increasing population size didnt have much effect on it, i raised the maxiter

x0 = np.random.uniform(-500, 500, num_dim)

sigma0 = 0.1 * (max_ind - min_ind)

# Giving CMA the same budget
total_evals = num_candidates * 7 * num_gen
cma_popsize = 100
cma_maxiter = total_evals // cma_popsize

opts = {
    "bounds": [-500, 500],
    "popsize": cma_popsize,
    "maxiter": cma_maxiter,
    "verb_disp": 0,
}

es = cma.CMAEvolutionStrategy(x0, sigma0, opts)

sigma_history = []

while not es.stop():
    solutions = es.ask()
    fitnesses = [schwefel(num_dim, x) for x in solutions]
    es.tell(solutions, fitnesses)

    sigma_history.append(es.sigma)

es.result_pretty()

plt.plot(sigma_history)
plt.xlabel("Generation")
plt.ylabel("Global sigma")
plt.title("CMA-ES Step Size Adaptation")
plt.grid(True)
plt.show()